# Electronics — turning a device into an amplifier

A transistor by itself has transconductance, not gain. It converts an input voltage into an output *current*, and a current is not yet useful. Gain appears only when that current is pushed through a resistance:

$$A_v=-g_m R_C$$

Everything in this notebook is a consequence of choosing what to put in that position and where to take the output from. The same device gives inverting voltage gain, unity-gain buffering, or difference amplification depending only on which terminal is the input and which is grounded.

Two themes run through all of it. The first is that **gain is never free** — every configuration trades it against bandwidth, linearity, impedance or headroom, and the last section shows that trade being made deliberately. The second is that the small-signal model from the previous notebook is doing all the work, so everything here inherits its validity limit: these results hold for signals small enough that the tangent still fits.

Schematics animate as before; the signal traces show the actual large-signal waveform, so clipping and distortion appear when they really occur rather than being assumed away.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import ipywidgets as widgets
from IPython.display import display

BG, PANEL, FG = "#05070b", "#0a0d14", "#c9cfda"
MUTED, GRIDC = "#6b7280", "#1b2130"
POS, NEG, DOT = "#3fd0c9", "#e0555c", "#ffd24a"
BLUE, ORANGE, GREEN, PURP = "#5aa9e6", "#e08a3c", "#7ddc7d", "#b48ce0"
VMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "volt", [(0.0, NEG), (0.5, "#4a5060"), (1.0, POS)])

plt.rcParams.update({
    "figure.dpi": 112, "font.size": 8.5, "axes.titlesize": 9,
    "figure.facecolor": BG, "savefig.facecolor": BG, "axes.facecolor": PANEL,
    "axes.edgecolor": GRIDC, "axes.labelcolor": FG, "text.color": FG,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRIDC,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.facecolor": PANEL, "legend.edgecolor": GRIDC, "legend.framealpha": 0.9,
})
SL = {"style": {"description_width": "104px"},
      "layout": widgets.Layout(width="290px"), "continuous_update": False}


def panel(ax, edge=None, lw=1.3):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(edge or GRIDC)
        s.set_linewidth(lw if edge else 0.8)
    ax.tick_params(colors=MUTED, labelsize=7)
    return ax


def readout(fig, x, y, lines, color=FG, size=7.4):
    fig.text(x, y, "\n".join(lines), family="monospace", fontsize=size,
             color=color, va="top", ha="left", linespacing=1.55)


def footer(fig, text):
    fig.text(0.010, 0.012, text, family="monospace", fontsize=6.6, color=MUTED)
    fig.text(0.990, 0.012, "electronics · amplifiers", family="monospace",
             fontsize=6.6, color=MUTED, ha="right")


def timeline(n, step=1, interval=90, desc="time"):
    p = widgets.Play(value=0, min=0, max=n, step=step, interval=interval)
    s = widgets.IntSlider(value=0, min=0, max=n, step=step, description=desc + ":",
                          continuous_update=False,
                          style={"description_width": "104px"},
                          layout=widgets.Layout(width="430px"))
    widgets.jslink((p, "value"), (s, "value"))
    return p, s


# ----- schematic primitives, Falstad style -------------------------------
def vcolor(v, vmax):
    return VMAP(np.clip(0.5 + 0.5 * v / max(vmax, 1e-9), 0, 1))


def wire(ax, pts, v, vmax, lw=2.6):
    pts = np.asarray(pts, float)
    seg = np.stack([pts[:-1], pts[1:]], axis=1)
    ax.add_collection(LineCollection(seg, colors=[vcolor(v, vmax)] * len(seg),
                                     linewidths=lw, zorder=2))


def node_dot(ax, p, v, vmax, s=34):
    ax.plot(*p, "o", ms=np.sqrt(s), color=vcolor(v, vmax), zorder=4)


def resistor(ax, p0, p1, v, vmax, label=None, n=6, amp=0.16):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.28, p1 - u * L * 0.28
    ts = np.linspace(0, 1, 2 * n + 1)
    zz = [a + (b - a) * t + nrm * amp * ((-1) ** k if 0 < k < 2 * n else 0)
          for k, t in enumerate(ts)]
    wire(ax, [p0, a], v, vmax)
    wire(ax, zz, v, vmax, lw=2.2)
    wire(ax, [b, p1], v, vmax)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.34), label, color=FG, fontsize=7.5,
                ha="center", va="center")


def capacitor(ax, p0, p1, v, vmax, label=None, gap=0.10, half=0.24):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * gap, c + u * gap
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    for q in (a, b):
        ax.plot(*np.stack([q - nrm * half, q + nrm * half]).T, color=FG, lw=2.4,
                zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def inductor(ax, p0, p1, v, vmax, label=None, coils=4, r=0.13):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.25, p1 - u * L * 0.25
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    seg = np.linalg.norm(b - a) / coils
    for k in range(coils):
        c = a + u * seg * (k + 0.5)
        th = np.linspace(0, np.pi, 24)
        pts = np.array([c + u * (seg / 2) * np.cos(np.pi - t) + nrm * r * np.sin(t)
                        for t in th])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=2.0, zorder=3)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.38), label, color=FG, fontsize=7.5,
                ha="center")


def diode(ax, p0, p1, v, vmax, label=None, s=0.20):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * s, c + u * s
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    ax.add_patch(mpatches.Polygon([a + nrm * s, a - nrm * s, b], closed=True,
                                  facecolor=ORANGE, edgecolor=ORANGE, zorder=3))
    ax.plot(*np.stack([b - nrm * s, b + nrm * s]).T, color=FG, lw=2.6, zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def source(ax, p0, p1, v, vmax, kind="dc", label=None, r=0.30):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    c = 0.5 * (p0 + p1)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    wire(ax, [p0, c - u * r], v, vmax); wire(ax, [c + u * r, p1], v, vmax)
    ax.add_patch(mpatches.Circle(c, r, fill=False, ec=FG, lw=2.0, zorder=3))
    if kind == "dc":
        ax.plot(*np.stack([c - u * 0.12 - nrm * 0.16, c - u * 0.12 + nrm * 0.16]).T,
                color=FG, lw=2.6, zorder=4)
        ax.plot(*np.stack([c + u * 0.12 - nrm * 0.09, c + u * 0.12 + nrm * 0.09]).T,
                color=FG, lw=2.0, zorder=4)
    else:
        t = np.linspace(-1, 1, 40)
        pts = np.array([c + u * (0.19 * t[i]) + nrm * 0.15 * np.sin(np.pi * t[i])
                        for i in range(len(t))])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=1.8, zorder=4)
    if label:
        ax.text(*(c + nrm * (r + 0.22)), label, color=FG, fontsize=7.5, ha="center")


def path_len(pts):
    p = np.asarray(pts, float)
    d = np.linalg.norm(np.diff(p, axis=0), axis=1)
    return np.r_[0, np.cumsum(d)]


def charge_dots(ax, loop, q, spacing=0.42, ms=4.2):
    """Yellow dots at arclength q + n*spacing — this is the current, visualised."""
    p = np.asarray(loop, float)
    s = path_len(p)
    L = s[-1]
    if L <= 0:
        return
    offs = (np.arange(0, L, spacing) + (q % spacing)) % L
    x = np.interp(offs, s, p[:, 0]); y = np.interp(offs, s, p[:, 1])
    ax.plot(x, y, "o", ms=ms, color=DOT, zorder=5, mec="none")


def sch_axes(ax, xlim, ylim):
    panel(ax)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    return ax


def loop_rect(x0, x1, y0, y1, n=60):
    top = np.stack([np.linspace(x0, x1, n), np.full(n, y1)], 1)
    right = np.stack([np.full(n, x1), np.linspace(y1, y0, n)], 1)
    bot = np.stack([np.linspace(x1, x0, n), np.full(n, y0)], 1)
    left = np.stack([np.full(n, x0), np.linspace(y0, y1, n)], 1)
    return np.vstack([top, right, bot, left])


def seg(p0, p1, n=40):
    return np.stack([np.linspace(p0[0], p1[0], n),
                     np.linspace(p0[1], p1[1], n)], 1)


VT = 0.02585          # kT/q at 300 K
IS_BJT = 1e-15
BETA = 150.0
VA = 50.0             # Early voltage
VTH_N = 1.0           # MOSFET threshold
KN = 2e-3             # MOSFET transconductance parameter, A/V^2
LAMBDA = 0.02


def bjt_ic(vbe, vce=5.0, Is=IS_BJT, va=VA):
    """Forward-active collector current with the Early effect."""
    ic = Is * np.exp(np.clip(vbe, -2, 1.2) / VT)
    return ic * (1 + np.maximum(vce, 0) / va)


def mos_id(vgs, vds, vth=VTH_N, k=KN, lam=LAMBDA):
    """Square-law NMOS: cutoff, triode, saturation."""
    vgs = np.asarray(vgs, float); vds = np.asarray(vds, float)
    vov = vgs - vth
    tri = k * (vov * vds - 0.5 * vds ** 2)
    sat = 0.5 * k * vov ** 2 * (1 + lam * vds)
    out = np.where(vds < vov, tri, sat)
    return np.where(vov <= 0, 0.0, np.maximum(out, 0.0))


def mos_region(vgs, vds, vth=VTH_N):
    if vgs - vth <= 0:
        return "cutoff"
    return "triode" if vds < vgs - vth else "saturation"


def transistor_npn(ax, p, v, vmax, label=None, s=0.42, flip=False):
    """NPN symbol: base left, collector up, emitter down."""
    p = np.asarray(p, float)
    ax.plot([p[0] - s * 0.55, p[0] - s * 0.55], [p[1] - s, p[1] + s],
            color=FG, lw=2.6, zorder=3)
    ax.plot([p[0] - s * 1.5, p[0] - s * 0.55], [p[1], p[1]], color=FG,
            lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.55, p[0] + s * 0.7], [p[1] + s * 0.45,
            p[1] + s * 1.25], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.55, p[0] + s * 0.7], [p[1] - s * 0.45,
            p[1] - s * 1.25], color=FG, lw=2.2, zorder=3)
    ax.add_patch(mpatches.Polygon(
        [[p[0] + s * 0.2, p[1] - s * 0.82], [p[0] + s * 0.05, p[1] - s * 0.45],
         [p[0] + s * 0.5, p[1] - s * 0.62]], closed=True, facecolor=FG,
        edgecolor=FG, zorder=4))
    if label:
        ax.text(p[0] + s * 1.05, p[1], label, color=FG, fontsize=8,
                va="center")


def transistor_nmos(ax, p, v, vmax, label=None, s=0.42):
    """NMOS symbol: gate left, drain up, source down."""
    p = np.asarray(p, float)
    ax.plot([p[0] - s * 0.95, p[0] - s * 0.95], [p[1] - s, p[1] + s],
            color=FG, lw=2.4, zorder=3)
    ax.plot([p[0] - s * 1.9, p[0] - s * 0.95], [p[1], p[1]], color=FG,
            lw=2.2, zorder=3)
    for dy in (-1, 0, 1):
        y0 = p[1] + dy * s * 0.62
        ax.plot([p[0] - s * 0.5, p[0] - s * 0.5],
                [y0 - s * 0.28, y0 + s * 0.28], color=FG, lw=2.4, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1] + s * 0.62,
            p[1] + s * 0.62], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] + s * 0.7, p[0] + s * 0.7], [p[1] + s * 0.62,
            p[1] + s * 1.3], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1] - s * 0.62,
            p[1] - s * 0.62], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] + s * 0.7, p[0] + s * 0.7], [p[1] - s * 0.62,
            p[1] - s * 1.3], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1], p[1]], color=FG,
            lw=2.0, zorder=3)
    ax.add_patch(mpatches.Polygon(
        [[p[0] + s * 0.35, p[1]], [p[0] + s * 0.05, p[1] + s * 0.2],
         [p[0] + s * 0.05, p[1] - s * 0.2]], closed=True, facecolor=FG,
        edgecolor=FG, zorder=4))
    if label:
        ax.text(p[0] + s * 1.25, p[1], label, color=FG, fontsize=8,
                va="center")


def hybrid_pi(Ic, beta=BETA, va=VA):
    """Small-signal parameters at an operating point."""
    gm = Ic / VT
    return dict(gm=gm, rpi=beta / gm, ro=va / max(Ic, 1e-15))


def par(*r):
    return 1.0 / sum(1.0 / x for x in r if x > 0)


def db(x):
    return 20 * np.log10(np.maximum(np.abs(x), 1e-12))


print("amplifier engine ready — hybrid-pi on top of the device models")
print(f"VT = {VT*1e3:.2f} mV   e-fold per VT   decade per {VT*np.log(10)*1e3:.2f} mV")
print(f"MOSFET Vth = {VTH_N:.2f} V   k = {KN*1e3:.2f} mA/V^2   VA = {VA:.0f} V")

## The common emitter — gain, inversion, and where it clips

Put a resistor in the collector and the signal current becomes a signal voltage. Because the current *pulls down* on the collector, the stage inverts:

$$A_v=-g_m\left(R_C\parallel r_o\right)$$

At 1 mA into 4.7 kΩ that is $-166$. The $r_o$ term matters: leaving it out gives $-181.8$, so the collector resistor and the device's own output resistance are genuinely in parallel — and differentiating the real exponential at the operating point confirms the $-181.8$ figure to **0.000%** when $r_o$ is excluded from both. The panels use the full expression throughout.

The waveform panel is where the honesty is. Push the input past a few millivolts and the output stops being a scaled copy: the top of the swing runs into the supply and the bottom into saturation, and because the device law is exponential the distortion is **asymmetric** long before either rail is reached. Watch the negative half of the output grow faster than the positive half shrinks.

Adding an emitter resistor changes the character of the stage completely:

$$A_v=\frac{-g_m(R_C\parallel r_o)}{1+g_mR_E}\;\approx\;-\frac{R_C}{R_E+1/g_m}
\;\xrightarrow{\ g_mR_E\gg1\ }\;-\frac{R_C}{R_E}$$

The middle form is exactly $-g_mR_C/(1+g_mR_E)$ rearranged — identical to six decimals — and it drops $r_o$, which is why the panel's number runs a little below it.

What matters is that the gain is now set by a **ratio of resistors** rather than a bias-dependent device parameter, so it is predictable, temperature-stable and far more linear. At $R_E=470\ \Omega$ it also raises the input impedance from 3.9 kΩ to 75 kΩ. That is the trade this notebook is about: 166 of fragile gain thrown away to get 8.7 you can rely on.

In [ ]:
def ce_stage(Ic, Rc, Re, Vcc):
    p = hybrid_pi(Ic)
    Av = -p["gm"] * par(Rc, p["ro"]) / (1 + p["gm"] * Re)
    Rin = p["rpi"] + (BETA + 1) * Re
    Rout = par(Rc, p["ro"])
    Vq = Vcc - Ic * Rc - Ic * Re
    return Av, Rin, Rout, Vq, p


def draw_ce(k, Ic_mA, Rc_k, Re, vin_mV, Vcc):
    Ic, Rc = Ic_mA * 1e-3, Rc_k * 1e3
    Av, Rin, Rout, Vq, p = ce_stage(Ic, Rc, Re, Vcc)
    Vbias = VT * np.log(Ic / IS_BJT) + Ic * Re
    t = np.linspace(0, 2, 600)
    A = vin_mV * 1e-3
    vin = A * np.sin(2 * np.pi * t)
    ic = np.zeros_like(t)
    for i, v in enumerate(vin):
        x = Ic
        for _ in range(60):
            ve = x * Re
            x = 0.7 * x + 0.3 * IS_BJT * np.exp(np.clip(
                (Vbias + v - ve), -2, 1.1) / VT)
        ic[i] = x
    vout = np.clip(Vcc - ic * Rc, 0.2, Vcc)
    vout_lin = Vq + Av * vin
    thd = np.max(np.abs(vout - vout_lin)) / max(np.max(np.abs(vout - Vq)), 1e-12) * 100
    kk = int(min(k, len(t) - 1))
    vmax = Vcc

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.3, 0.55],
                          wspace=0.3, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 3.8), (-0.5, 3.8))
    transistor_npn(a0, (2.0, 1.8), 0.0, vmax, None)
    resistor(a0, (2.3, 3.4), (2.3, 2.6), Vcc, vmax, f"Rc {Rc_k:.1f}k")
    wire(a0, [(2.3, 2.6), (2.3, 1.8 + 0.53)], vout[kk], vmax)
    wire(a0, [(2.3, 3.4), (0.7, 3.4)], Vcc, vmax)
    wire(a0, [(2.3, 2.6), (3.4, 2.6)], vout[kk], vmax)
    node_dot(a0, (3.4, 2.6), vout[kk], vmax)
    a0.text(3.5, 2.6, "out", color=FG, fontsize=8)
    if Re > 0:
        wire(a0, [(2.3, 1.8 - 0.53), (2.3, 1.0)], ic[kk] * Re, vmax)
        resistor(a0, (2.3, 1.0), (2.3, 0.2), ic[kk] * Re / 2, vmax, f"Re {Re:.0f}")
    else:
        wire(a0, [(2.3, 1.8 - 0.53), (2.3, 0.2)], 0.0, vmax)
    wire(a0, [(2.3, 0.2), (0.7, 0.2)], 0.0, vmax)
    wire(a0, [(2.0 - 0.63, 1.8), (1.1, 1.8)], Vbias + vin[kk], vmax)
    source(a0, (0.7, 0.2), (0.7, 3.4), Vcc / 2, vmax, "dc", f"{Vcc:.0f}V")
    source(a0, (1.1, 0.2), (1.1, 1.8), (Vbias + vin[kk]) / 2, vmax, "ac", "in")
    charge_dots(a0, seg((2.3, 3.4), (2.3, 1.8 + 0.53), 24), k / 120 * ic[kk] * 3e4,
                spacing=0.24, ms=3.6)
    a0.set_title(f"$A_v$ = {Av:.2f}  ·  the collector pulls down, so it inverts",
                 fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(t, (Vbias + vin) * 1e3 - Vbias * 1e3, color=POS, lw=1.4,
            label=f"input ±{vin_mV:.1f} mV")
    a1b = a1.twinx()
    a1b.plot(t, vout, color=DOT, lw=1.6, label="output")
    a1b.plot(t, vout_lin, color=MUTED, lw=1.0, ls="--", label="ideal linear")
    a1b.axhline(Vcc, color=NEG, lw=0.8, ls=":")
    a1b.set_ylabel("out  (V)", color=DOT); a1b.tick_params(colors=DOT, labelsize=7)
    a1b.grid(False)
    a1.axvline(t[kk], color=FG, lw=1.0, ls=":")
    a1.set_ylabel("in  (mV)", color=POS); a1.tick_params(axis="y", colors=POS)
    a1.set_xlabel("cycles")
    a1.set_title(f"distortion {thd:.2f}%  ·  the exponential bends one half first")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    res = np.logspace(0, 3.5, 300)
    Avs = np.array([abs(ce_stage(Ic, Rc, r, Vcc)[0]) for r in res])
    Rins = np.array([ce_stage(Ic, Rc, r, Vcc)[1] for r in res])
    a2.loglog(res, Avs, color=ORANGE, lw=1.7, label="|Av|")
    a2.loglog(res, Rc / res, color=MUTED, lw=1.0, ls="--", label="Rc/Re")
    a2.axvline(max(Re, 1), color=FG, lw=1.0, ls="--")
    a2b = a2.twinx()
    a2b.loglog(res, Rins / 1e3, color=PURP, lw=1.4)
    a2b.set_ylabel("Rin  (kΩ)", color=PURP); a2b.tick_params(colors=PURP, labelsize=7)
    a2b.grid(False)
    a2.set_xlabel("emitter resistor  (Ω)"); a2.set_ylabel("|gain|")
    a2.legend(fontsize=7)
    a2.set_title("degeneration: gain down, impedance up, linearity up")

    readout(fig, 0.845, 0.90, [
        "OPERATING POINT", "─" * 26,
        f"Ic          {Ic_mA:>10.3f}mA",
        f"Vcc         {Vcc:>10.1f}V",
        f"Rc          {Rc_k:>10.2f}kΩ",
        f"Re          {Re:>10.1f}Ω",
        f"Vq out      {Vq:>10.3f}V",
        "", "SMALL SIGNAL", "─" * 26,
        f"gm          {p['gm']*1e3:>10.3f}mS",
        f"rpi         {p['rpi']/1e3:>10.3f}kΩ",
        f"ro          {p['ro']/1e3:>10.2f}kΩ",
        f"gm·Re       {p['gm']*Re:>10.3f}",
        "", "STAGE", "─" * 26,
        f"Av          {Av:>+10.3f}",
        f"in dB       {20*np.log10(abs(Av)):>10.2f}dB",
        f"−Rc/Re      {(-Rc/Re if Re>0 else float('nan')):>+10.3f}",
        f"Rin         {Rin/1e3:>10.2f}kΩ",
        f"Rout        {Rout/1e3:>10.2f}kΩ",
        "", "SWING", "─" * 26,
        f"input       {vin_mV:>10.2f}mV",
        f"output      {np.ptp(vout)/2:>10.3f}V",
        f"distortion  {thd:>10.2f}%",
        f"headroom up {Vcc-Vq:>10.3f}V",
    ], color=NEG if thd > 10 else FG)
    footer(fig, f"Av = −gm(Rc||ro) = {Av:.2f}   ·   with degeneration → −Rc/Re   ·   "
                f"gain traded for predictability")
    plt.show()


_p1, _s1 = timeline(119, step=2)
w1 = dict(Ic_mA=widgets.FloatSlider(value=1.0, min=0.1, max=4.0, step=0.05,
                                    description="Ic (mA):", **SL),
          Rc_k=widgets.FloatSlider(value=4.7, min=0.5, max=15, step=0.1,
                                   description="Rc (kΩ):", **SL),
          Re=widgets.FloatSlider(value=0, min=0, max=1000, step=10,
                                 description="Re (Ω):", **SL),
          vin_mV=widgets.FloatSlider(value=2, min=0.2, max=40, step=0.2,
                                     description="input (mV):", **SL),
          Vcc=widgets.FloatSlider(value=10, min=5, max=20, step=1,
                                  description="Vcc (V):", **SL),
          k=_s1)
display(widgets.VBox([widgets.HBox([w1["Ic_mA"], w1["Rc_k"], w1["Re"]]),
                      widgets.HBox([w1["vin_mV"], w1["Vcc"]]),
                      widgets.HBox([_p1, _s1])]),
        widgets.interactive_output(draw_ce, w1))

## The emitter follower — no voltage gain, and that is the point

Take the output from the emitter instead and the voltage gain collapses to just under one:

$$A_v=\frac{g_mR_E}{1+g_mR_E}=0.9748\ \text{at 1 mA into 1 k}\Omega$$

A stage that does not amplify sounds useless. What it does instead is **transform impedance**, and by an enormous factor:

$$R_{in}=r_\pi+(\beta+1)R_E=155\ \text{k}\Omega,\qquad
R_{out}=\left(\frac{1}{g_m}+\frac{R_S}{\beta+1}\right)\parallel R_E$$

At 1 mA driven from 1 kΩ that is 155 kΩ in and 31 Ω out — a ratio of about **5000:1**. (The familiar shorthand $1/g_m+R_S/(\beta+1)$ gives 32 Ω; the emitter resistor really is in parallel at the output, and the difference grows to 26% by $R_S=50$ kΩ.) The follower presents a light load to whatever drives it and a stiff source to whatever it drives, which is exactly what the common-emitter stage above needs on both sides — its own input impedance is only 3.9 kΩ and its output impedance is several kΩ, so cascading two CE stages directly means most of the signal is lost in the interface.

The output resistance formula is worth reading carefully. $R_S/(\beta+1)$ means a *source* impedance is divided by β when seen at the emitter — the same transformation running backwards. Drag the source impedance and watch $R_{out}$ climb: a follower driven from 50 kΩ has 263 Ω of output impedance, not 26 Ω. A buffer does not erase the source impedance, it divides it.

The voltage gain never reaches one because $1/g_m$ is in series with $R_E$, and it falls further as the load gets heavier. Load the output and watch both the gain and the useful swing shrink.

In [ ]:
def follower(Ic, Re, RL, Rs):
    p = hybrid_pi(Ic)
    Reff = par(Re, RL) if RL > 0 else Re
    Av = (p["gm"] * Reff) / (1 + p["gm"] * Reff)
    Rin = p["rpi"] + (BETA + 1) * Reff
    Rout = par(1 / p["gm"] + Rs / (BETA + 1), Re)
    return Av, Rin, Rout, p, Reff


def draw_follower(k, Ic_mA, Re, RL_k, Rs_k, Vcc):
    Ic, RL, Rs = Ic_mA * 1e-3, RL_k * 1e3, Rs_k * 1e3
    Av, Rin, Rout, p, Reff = follower(Ic, Re, RL, Rs)
    t = np.linspace(0, 2, 500)
    vin = 1.0 * np.sin(2 * np.pi * t)
    vout = Av * vin
    kk = int(min(k, len(t) - 1))
    vmax = Vcc

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.3, 0.55],
                          wspace=0.3, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.2), (-0.5, 3.6))
    transistor_npn(a0, (2.0, 2.0), 0.0, vmax, None)
    wire(a0, [(2.0 + 0.29, 2.0 + 0.53), (2.3, 3.2)], Vcc, vmax)
    wire(a0, [(2.3, 3.2), (0.7, 3.2)], Vcc, vmax)
    wire(a0, [(2.0 + 0.29, 2.0 - 0.53), (2.3, 1.2)], vout[kk] + 3, vmax)
    resistor(a0, (2.3, 1.2), (2.3, 0.2), (vout[kk] + 3) / 2, vmax, f"Re {Re:.0f}")
    wire(a0, [(2.3, 1.2), (3.4, 1.2)], vout[kk] + 3, vmax)
    if RL_k < 900:
        resistor(a0, (3.4, 1.2), (3.4, 0.2), (vout[kk] + 3) / 2, vmax,
                 f"RL {RL_k:.1f}k")
    node_dot(a0, (3.4, 1.2), vout[kk] + 3, vmax)
    a0.text(3.55, 1.45, "out", color=FG, fontsize=8)
    wire(a0, [(0.7, 0.2), (3.4, 0.2)], 0.0, vmax)
    resistor(a0, (1.05, 2.0), (2.0 - 0.63, 2.0), vin[kk] + 3.7, vmax,
             f"Rs {Rs_k:.0f}k")
    source(a0, (0.7, 0.2), (0.7, 3.2), Vcc / 2, vmax, "dc", f"{Vcc:.0f}V")
    source(a0, (1.05, 0.2), (1.05, 2.0), (vin[kk] + 3.7) / 2, vmax, "ac", "in")
    charge_dots(a0, seg((2.3, 3.2), (2.0 + 0.29, 2.0 + 0.53), 20), k / 120 * Ic * 3e4,
                spacing=0.24, ms=3.6)
    charge_dots(a0, seg((2.3, 1.2), (2.3, 0.3), 20), k / 120 * Ic * 3e4,
                spacing=0.24, ms=3.6)
    a0.set_title("output follows the input, one diode drop below", fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(t, vin, color=POS, lw=1.5, label="input")
    a1.plot(t, vout, color=DOT, lw=1.5, label="output")
    a1.axvline(t[kk], color=FG, lw=1.0, ls=":")
    a1.set_ylabel("volts"); a1.legend(fontsize=7)
    a1.set_title(f"$A_v$ = {Av:.5f} — close to one, never equal to it")

    a2 = panel(fig.add_subplot(gs[1, 1]), PURP)
    rs = np.logspace(1, 5.3, 300)
    ro = np.array([follower(Ic, Re, RL, r)[2] for r in rs])
    a2.loglog(rs / 1e3, ro, color=PURP, lw=1.7, label="$R_{out}$")
    a2.axhline(1 / p["gm"], color=MUTED, lw=0.9, ls=":")
    a2.text(rs[3] / 1e3, 1 / p["gm"] * 1.15, "$1/g_m$", color=MUTED, fontsize=7.5)
    a2.axvline(Rs_k, color=FG, lw=1.0, ls="--")
    a2.plot([Rs_k], [Rout], "o", ms=7, color=DOT)
    a2.set_xlabel("source impedance  (kΩ)"); a2.set_ylabel("Ω")
    a2.legend(fontsize=7)
    a2.set_title("source impedance divided by β — but only divided, not removed")

    readout(fig, 0.845, 0.90, [
        "STAGE", "─" * 26,
        f"Ic          {Ic_mA:>10.3f}mA",
        f"Re          {Re:>10.1f}Ω",
        f"RL          {RL_k:>10.2f}kΩ",
        f"Re||RL      {Reff:>10.1f}Ω",
        f"Rs          {Rs_k:>10.2f}kΩ",
        "", "SMALL SIGNAL", "─" * 26,
        f"gm          {p['gm']*1e3:>10.3f}mS",
        f"1/gm        {1/p['gm']:>10.3f}Ω",
        f"rpi         {p['rpi']/1e3:>10.3f}kΩ",
        "", "RESULT", "─" * 26,
        f"Av          {Av:>10.5f}",
        f"loss        {20*np.log10(Av):>+10.3f}dB",
        f"Rin         {Rin/1e3:>10.2f}kΩ",
        f"Rout        {Rout:>10.2f}Ω",
        f"Rin/Rout    {Rin/Rout:>10.0f}×",
        "", "no voltage gain",
        "large current gain",
        "that is the whole job",
    ])
    footer(fig, f"Av = gmRe/(1+gmRe)   ·   Rin = rπ+(β+1)Re   ·   "
                f"Rout = 1/gm + Rs/(β+1)   ·   ratio {Rin/Rout:.0f}×")
    plt.show()


_p2, _s2 = timeline(119, step=2)
w2 = dict(Ic_mA=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1,
                                    description="Ic (mA):", **SL),
          Re=widgets.FloatSlider(value=1000, min=100, max=5000, step=50,
                                 description="Re (Ω):", **SL),
          RL_k=widgets.FloatSlider(value=1000, min=0.1, max=1000, step=0.1,
                                   description="RL (kΩ):", **SL),
          Rs_k=widgets.FloatSlider(value=1, min=0.1, max=100, step=0.5,
                                   description="Rs (kΩ):", **SL),
          Vcc=widgets.FloatSlider(value=10, min=5, max=20, step=1,
                                  description="Vcc (V):", **SL),
          k=_s2)
display(widgets.VBox([widgets.HBox([w2["Ic_mA"], w2["Re"], w2["RL_k"]]),
                      widgets.HBox([w2["Rs_k"], w2["Vcc"]]),
                      widgets.HBox([_p2, _s2])]),
        widgets.interactive_output(draw_follower, w2))

## The differential pair — a current split two ways

Two matched transistors share a tail current. Whatever the inputs do individually, the two collector currents must always add to $I_{tail}$, so the only thing the pair responds to is the **difference**:

$$I_1=\frac{I_{tail}}{1+e^{-v_{id}/V_T}},\qquad
I_1-I_2=I_{tail}\tanh\!\left(\frac{v_{id}}{2V_T}\right)$$

The tanh is the whole behaviour. It is linear over about $\pm V_T$ — measured 0.31% error at ±5 mV and 1.24% at ±10 mV — then compresses, and by ±100 mV the pair is **fully switched**, with 97.95% of the tail current in one side. That switching is not a defect; it is exactly what a logic gate or a mixer wants.

The transconductance at balance is $I_{tail}/2V_T$ differentially, which is just the $g_m$ of each half at $I_{tail}/2$. Note what the tail current does: raise it and you get more gain but a *narrower* linear range in absolute terms, because the tanh's argument does not care about current at all.

The reason this circuit opens every op-amp is the right panel. A signal common to both inputs tries to change the tail node, and the tail source resists it, so common-mode gain is suppressed by roughly $1/(2g_mR_{tail})$. Measured CMRR climbs from **32 dB** with a 1 kΩ tail resistor to **92 dB** with 1 MΩ — which is precisely why the tail is a current mirror from the previous notebook rather than a resistor.

In [ ]:
def diffpair(Itail, vid, Rc, Rtail):
    i1 = Itail / (1 + np.exp(-np.clip(vid, -0.4, 0.4) / VT))
    i2 = Itail - i1
    return i1, i2


def draw_diffpair(k, Itail_mA, vid_mV, Rc_k, Rtail_k, Vcc):
    Itail, Rc, Rtail = Itail_mA * 1e-3, Rc_k * 1e3, Rtail_k * 1e3
    vid = vid_mV * 1e-3
    i1, i2 = diffpair(Itail, vid, Rc, Rtail)
    gm_pair = Itail / (2 * VT)
    Adm = gm_pair * Rc
    Acm = Rc / (2 * Rtail + 1 / gm_pair)
    cmrr = 20 * np.log10(Adm / max(Acm, 1e-15))
    vv = np.linspace(-0.15, 0.15, 500)
    I1 = Itail / (1 + np.exp(-vv / VT))
    kk = k / 120
    vmax = Vcc

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1.25, 0.55],
                          wspace=0.3, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.6), (-0.8, 3.8))
    wire(a0, [(0.5, 3.4), (4.2, 3.4)], Vcc, vmax)
    resistor(a0, (1.5, 3.4), (1.5, 2.6), Vcc, vmax, f"Rc")
    resistor(a0, (3.5, 3.4), (3.5, 2.6), Vcc, vmax, f"Rc")
    transistor_npn(a0, (1.8, 1.9), 0.0, vmax, None)
    transistor_npn(a0, (3.2, 1.9), 0.0, vmax, None)
    v1 = Vcc - i1 * Rc; v2 = Vcc - i2 * Rc
    wire(a0, [(1.5, 2.6), (1.8 + 0.29, 1.9 + 0.53)], v1, vmax)
    wire(a0, [(3.5, 2.6), (3.2 + 0.29, 1.9 + 0.53)], v2, vmax)
    wire(a0, [(1.8 + 0.29, 1.9 - 0.53), (1.8 + 0.29, 1.0)], 0.5, vmax)
    wire(a0, [(3.2 + 0.29, 1.9 - 0.53), (3.2 + 0.29, 1.0)], 0.5, vmax)
    wire(a0, [(1.8 + 0.29, 1.0), (3.2 + 0.29, 1.0)], 0.5, vmax)
    resistor(a0, (2.5, 1.0), (2.5, 0.1), 0.3, vmax, f"tail")
    wire(a0, [(0.5, 0.1), (4.2, 0.1)], 0.0, vmax)
    wire(a0, [(1.8 - 0.63, 1.9), (1.0, 1.9)], vid / 2 + 3, vmax)
    wire(a0, [(3.2 - 0.63, 1.9), (2.75, 1.9)], -vid / 2 + 3, vmax)
    a0.text(0.75, 2.15, f"+{vid_mV/2:.1f} mV", color=POS, fontsize=7)
    a0.text(2.5, 2.2, f"−{vid_mV/2:.1f} mV", color=PURP, fontsize=7)
    node_dot(a0, (1.5, 2.6), v1, vmax); node_dot(a0, (3.5, 2.6), v2, vmax)
    charge_dots(a0, seg((1.5, 3.4), (1.8 + 0.29, 1.9 + 0.53), 22), kk * i1 * 4e4,
                spacing=0.22, ms=3.4)
    charge_dots(a0, seg((3.5, 3.4), (3.2 + 0.29, 1.9 + 0.53), 22), kk * i2 * 4e4,
                spacing=0.22, ms=3.4)
    charge_dots(a0, seg((2.5, 1.0), (2.5, 0.2), 18), kk * Itail * 4e4,
                spacing=0.22, ms=3.4)
    a0.set_title(f"the tail is fixed — {i1*1e6:.1f} µA one side, "
                 f"{i2*1e6:.1f} µA the other", fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(vv * 1e3, I1 * 1e6, color=POS, lw=1.7, label="$I_1$")
    a1.plot(vv * 1e3, (Itail - I1) * 1e6, color=PURP, lw=1.7, label="$I_2$")
    a1.plot(vv * 1e3, (Itail / 2 + gm_pair / 2 * vv) * 1e6, color=MUTED, lw=1.0,
            ls="--", label="linear")
    a1.axvline(vid_mV, color=FG, lw=1.0, ls="--")
    a1.axvspan(-VT * 1e3, VT * 1e3, color=GREEN, alpha=0.10)
    a1.set_ylim(0, Itail * 1.15e6)
    a1.set_xlabel("$v_{id}$  (mV)"); a1.set_ylabel("µA"); a1.legend(fontsize=7)
    a1.set_title(f"tanh — linear within ±$V_T$, switched by ±{4*VT*1e3:.0f} mV")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    rt = np.logspace(2, 6.5, 300)
    cm = 20 * np.log10(Adm / (Rc / (2 * rt + 1 / gm_pair)))
    a2.semilogx(rt / 1e3, cm, color=ORANGE, lw=1.7)
    a2.axvline(Rtail_k, color=FG, lw=1.0, ls="--")
    a2.plot([Rtail_k], [cmrr], "o", ms=7, color=DOT)
    a2.set_xlabel("tail resistance  (kΩ)"); a2.set_ylabel("CMRR  (dB)")
    a2.set_title("a current-source tail is why op-amps reject common mode")

    readout(fig, 0.845, 0.90, [
        "PAIR", "─" * 26,
        f"Itail       {Itail_mA:>10.3f}mA",
        f"Rc          {Rc_k:>10.2f}kΩ",
        f"Rtail       {Rtail_k:>10.1f}kΩ",
        "", "SPLIT", "─" * 26,
        f"vid         {vid_mV:>+10.2f}mV",
        f"I1          {i1*1e6:>10.3f}µA",
        f"I2          {i2*1e6:>10.3f}µA",
        f"I1+I2       {(i1+i2)*1e6:>10.3f}µA",
        f"fraction    {i1/Itail:>10.4f}",
        f"vid/VT      {vid/VT:>+10.3f}",
        "", "GAIN", "─" * 26,
        f"gm pair     {gm_pair*1e3:>10.3f}mS",
        f"= Itail/2VT",
        f"Adm         {Adm:>10.2f}",
        f"Acm         {Acm:>10.5f}",
        f"CMRR        {cmrr:>10.2f}dB",
        "", "LINEARITY", "─" * 26,
        " ±5 mV → 0.31%",
        "±10 mV → 1.24%",
        "±26 mV → 8.29%",
    ], color=ORANGE if abs(vid_mV) > 26 else FG)
    footer(fig, f"I1−I2 = Itail·tanh(vid/2VT)   ·   gm = Itail/2VT   ·   "
                f"CMRR ≈ 2gm·Rtail = {cmrr:.1f} dB")
    plt.show()


_p3, _s3 = timeline(119, step=2)
w3 = dict(Itail_mA=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1,
                                       description="Itail (mA):", **SL),
          vid_mV=widgets.FloatSlider(value=10, min=-120, max=120, step=1,
                                     description="vid (mV):", **SL),
          Rc_k=widgets.FloatSlider(value=4.7, min=0.5, max=20, step=0.1,
                                   description="Rc (kΩ):", **SL),
          Rtail_k=widgets.FloatSlider(value=100, min=1, max=2000, step=1,
                                      description="Rtail (kΩ):", **SL),
          Vcc=widgets.FloatSlider(value=10, min=5, max=20, step=1,
                                  description="Vcc (V):", **SL),
          k=_s3)
display(widgets.VBox([widgets.HBox([w3["Itail_mA"], w3["vid_mV"], w3["Rc_k"]]),
                      widgets.HBox([w3["Rtail_k"], w3["Vcc"]]),
                      widgets.HBox([_p3, _s3])]),
        widgets.interactive_output(draw_diffpair, w3))

## The Miller effect — why gain costs bandwidth

A few picofarads sit between the base and collector of every transistor. That capacitor is bridging a node that swings one way and a node that swings the *opposite* way with $|A_v|$ times the amplitude, so the charge it demands is multiplied:

$$C_{in}=C_\mu(1-A_v)$$

With $A_v=-200$ and $C_\mu=5$ pF the input sees **1005 pF** — two hundred times the physical component. The stage then rolls off against the source impedance far earlier than the device itself would.

The consequence is the conserved quantity that dominates amplifier design. Measured across three gains driven from 1 kΩ: $|A_v|=10$ gives 2.27 MHz, $|A_v|=50$ gives 0.589 MHz, $|A_v|=200$ gives 0.156 MHz — and the products are 22.7, 29.5 and 31.2 MHz. **Gain and bandwidth trade almost one for one**, which is why a single stage cannot simply be made better and why op-amps quote a gain–bandwidth product rather than a gain.

Two escapes exist and both appear later. Drive the stage from a lower impedance and the same $C_{in}$ matters less — the follower from two sections ago is the standard answer. Or stop the collector from swinging at all, which is what a cascode does: the Miller multiplier is $(1-A_v)$, so if the transistor's own collector sees a gain of about $-1$, the multiplication disappears.

In [ ]:
def miller_response(f, Av, Cmu, Cpi, Rs):
    Cin = Cmu * (1 - Av) + Cpi
    return Av / (1 + 1j * 2 * np.pi * f * Rs * Cin), Cin


def draw_miller(Av_mag, Cmu_pF, Cpi_pF, Rs_k):
    Cmu, Cpi, Rs = Cmu_pF * 1e-12, Cpi_pF * 1e-12, Rs_k * 1e3
    Av = -Av_mag
    ff = np.logspace(3, 9, 500)
    H, Cin = miller_response(ff, Av, Cmu, Cpi, Rs)
    f3 = 1 / (2 * np.pi * Rs * Cin)
    gbw = Av_mag * f3
    gains = np.logspace(0.3, 2.7, 60)
    f3s = np.array([1 / (2 * np.pi * Rs * (Cmu * (1 + g) + Cpi)) for g in gains])
    vmax = 1.0

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.35, 0.55],
                          wspace=0.3, hspace=0.46, left=0.03, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 3.8), (-0.5, 3.6))
    transistor_npn(a0, (2.0, 1.8), 0.0, vmax, None)
    resistor(a0, (2.3, 3.2), (2.3, 2.5), 1.0, vmax, "Rc")
    wire(a0, [(2.3, 2.5), (2.3, 1.8 + 0.53)], -0.6, vmax)
    wire(a0, [(2.3, 3.2), (0.7, 3.2)], 1.0, vmax)
    wire(a0, [(2.0 + 0.29, 1.8 - 0.53), (2.3, 0.2)], 0.0, vmax)
    wire(a0, [(0.7, 0.2), (2.6, 0.2)], 0.0, vmax)
    resistor(a0, (1.0, 1.8), (2.0 - 0.63, 1.8), 0.5, vmax, f"Rs {Rs_k:.1f}k")
    capacitor(a0, (2.0 - 0.63, 1.8), (2.3, 2.5), 0.2, vmax, f"Cμ {Cmu_pF:.1f}p")
    a0.annotate("", xy=(1.55, 2.35), xytext=(1.55, 1.55),
                arrowprops=dict(arrowstyle="<->", color=NEG, lw=1.4))
    a0.text(1.15, 2.5, f"×{1-Av:.0f}", color=NEG, fontsize=9)
    source(a0, (0.7, 0.2), (0.7, 1.8), 0.4, vmax, "ac", "in")
    node_dot(a0, (2.3, 2.5), -0.6, vmax)
    a0.set_title("one small capacitor, bridging two nodes that move oppositely",
                 fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.semilogx(ff, db(H), color=POS, lw=1.8)
    for g in (10, 50, 200):
        Hh, Cc = miller_response(ff, -g, Cmu, Cpi, Rs)
        a1.semilogx(ff, db(Hh), color=MUTED, lw=0.8, alpha=0.5)
    a1.axhline(db(np.array([Av]))[0] - 3.01, color=MUTED, lw=0.8, ls=":")
    a1.axvline(f3, color=DOT, lw=1.1, ls="--")
    a1.text(f3 * 1.2, db(np.array([Av]))[0] - 10, f"{f3/1e6:.3f} MHz", color=DOT,
            fontsize=7.5)
    a1.set_ylabel("|gain|  (dB)"); a1.set_ylim(-20, 55)
    a1.set_title(f"more gain, less bandwidth — GBW = {gbw/1e6:.2f} MHz")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    a2.loglog(gains, f3s / 1e6, color=ORANGE, lw=1.7, label="$f_{3dB}$")
    a2.loglog(gains, gains * f3s / 1e6, color=PURP, lw=1.7, label="GBW")
    a2.axvline(Av_mag, color=FG, lw=1.0, ls="--")
    a2.set_xlabel("|gain|"); a2.set_ylabel("MHz"); a2.legend(fontsize=7)
    a2.set_title("bandwidth falls as gain rises — the product barely moves")

    readout(fig, 0.845, 0.90, [
        "STAGE", "─" * 26,
        f"|Av|        {Av_mag:>10.1f}",
        f"in dB       {20*np.log10(Av_mag):>10.2f}dB",
        f"Rs          {Rs_k:>10.2f}kΩ",
        "", "CAPACITANCE", "─" * 26,
        f"Cμ physical {Cmu_pF:>10.2f}pF",
        f"Cπ          {Cpi_pF:>10.2f}pF",
        f"multiplier  {1-Av:>10.1f}×",
        f"Miller Cμ   {Cmu*(1-Av)*1e12:>10.2f}pF",
        f"total Cin   {Cin*1e12:>10.2f}pF",
        "", "BANDWIDTH", "─" * 26,
        f"f3dB        {f3/1e6:>10.4f}MHz",
        f"GBW         {gbw/1e6:>10.3f}MHz",
        "", "REFERENCE", "─" * 26,
        "|Av|= 10 → 2.27 MHz",
        "|Av|= 50 → 0.59 MHz",
        "|Av|=200 → 0.16 MHz",
        "products 22.7 / 29.5",
        "         / 31.2 MHz",
        "", "cascode removes the",
        "multiplication entirely",
    ], color=NEG if Av_mag > 100 else FG)
    footer(fig, f"Cin = Cμ(1−Av) = {Cin*1e12:.1f} pF   ·   "
                f"f3dB = 1/(2πRsCin)   ·   gain × bandwidth ≈ constant")
    plt.show()


w4 = dict(Av_mag=widgets.FloatSlider(value=50, min=2, max=400, step=2,
                                     description="|gain|:", **SL),
          Cmu_pF=widgets.FloatSlider(value=5, min=0.5, max=20, step=0.5,
                                     description="Cμ (pF):", **SL),
          Cpi_pF=widgets.FloatSlider(value=15, min=1, max=50, step=1,
                                     description="Cπ (pF):", **SL),
          Rs_k=widgets.FloatSlider(value=1, min=0.1, max=20, step=0.1,
                                   description="Rs (kΩ):", **SL))
display(widgets.HBox([w4["Av_mag"], w4["Cmu_pF"], w4["Cpi_pF"], w4["Rs_k"]]),
        widgets.interactive_output(draw_miller, w4))

## Negative feedback — giving away gain to buy everything else

Feed a fraction $\beta$ of the output back in opposition and the closed-loop gain becomes

$$A_{cl}=\frac{A}{1+A\beta}\;\xrightarrow{\ A\beta\gg1\ }\;\frac{1}{\beta}$$

The open-loop gain has **dropped out**. The result is set by the feedback network, which can be two resistors, and the amplifier's own gain matters only through the loop gain $A\beta$ that decides how completely it drops out.

The desensitivity is the point, and it is measurable. With $A=10^5$ and $\beta=0.01$, changing the open-loop gain by **+50%** — a transistor swap, a temperature shift, whatever — moves the closed-loop gain by **+0.033%**. The circuit stops caring about the device.

The same division applies to everything else the loop encloses. Distortion generated inside the loop is reduced by $1+A\beta$, output impedance falls by the same factor, input impedance rises by it, and bandwidth is multiplied by it. That last one is why the gain-bandwidth product from the previous section is the resource being spent: you can have gain, or bandwidth, and feedback lets you choose the split rather than accept whatever the device happened to give.

The error term is worth keeping in view. The closed-loop gain is short of the ideal $1/\beta$ by about $1/(A\beta)$ — measured $-0.99\%$ at a loop gain of 100, $-0.0999\%$ at 1000, $-0.01\%$ at 10 000. A real op-amp's gain error is nothing more than this.

In [ ]:
def draw_feedback(A0_dB, beta_pct, fp_kHz, dA_pct):
    A0 = 10 ** (A0_dB / 20)
    beta_f = beta_pct / 100
    fp = fp_kHz * 1e3
    ff = np.logspace(0, 8, 600)
    A = A0 / (1 + 1j * ff / fp)
    Acl = A / (1 + A * beta_f)
    ideal = 1 / beta_f
    loop = A0 * beta_f
    err = (A0 / (1 + loop) - ideal) / ideal * 100
    A0b = A0 * (1 + dA_pct / 100)
    Aclb = A0b / (1 + A0b * beta_f)
    dcl = (Aclb - A0 / (1 + loop)) / (A0 / (1 + loop)) * 100
    f3 = fp * (1 + loop)

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.35, 0.55],
                          wspace=0.3, hspace=0.46, left=0.03, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = panel(fig.add_subplot(gs[:, 0]))
    a0.set_xlim(0, 10); a0.set_ylim(0, 10)
    a0.set_xticks([]); a0.set_yticks([]); a0.grid(False)
    a0.add_patch(mpatches.Circle((2.4, 6.6), 0.55, fill=False, ec=FG, lw=1.8))
    a0.text(2.4, 6.6, "+", color=FG, fontsize=13, ha="center", va="center")
    a0.text(2.4, 5.75, "−", color=NEG, fontsize=13, ha="center", va="center")
    a0.add_patch(mpatches.Polygon([[4.0, 5.4], [4.0, 7.8], [6.6, 6.6]],
                                  closed=True, fill=False, ec=POS, lw=2.0))
    a0.text(4.9, 6.6, "A", color=POS, fontsize=12, ha="center", va="center")
    a0.add_patch(mpatches.Rectangle((4.2, 2.4), 2.2, 1.2, fill=False, ec=ORANGE,
                                    lw=2.0))
    a0.text(5.3, 3.0, "β", color=ORANGE, fontsize=12, ha="center", va="center")
    for xy, xytext in ((( 2.4, 6.6), (0.6, 6.6)), ((4.0, 6.6), (2.95, 6.6)),
                       ((8.6, 6.6), (6.6, 6.6))):
        a0.annotate("", xy=xy, xytext=xytext,
                    arrowprops=dict(arrowstyle="-|>", color=FG, lw=1.6))
    a0.plot([7.6, 7.6, 6.4], [6.6, 3.0, 3.0], color=FG, lw=1.6)
    a0.annotate("", xy=(6.4, 3.0), xytext=(7.0, 3.0),
                arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=1.6))
    a0.plot([4.2, 2.4, 2.4], [3.0, 3.0, 6.05], color=ORANGE, lw=1.6)
    a0.annotate("", xy=(2.4, 6.05), xytext=(2.4, 4.6),
                arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=1.6))
    a0.text(0.5, 7.0, "in", color=FG, fontsize=9)
    a0.text(8.3, 7.0, "out", color=FG, fontsize=9)
    a0.text(5.0, 1.6, f"loop gain Aβ = {loop:,.0f}", color=DOT, fontsize=9,
            ha="center")
    a0.set_title("the output is compared with the input, and the error is amplified",
                 fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.semilogx(ff, db(A), color=MUTED, lw=1.4, label="open loop")
    a1.semilogx(ff, db(Acl), color=POS, lw=1.9, label="closed loop")
    a1.axhline(20 * np.log10(ideal), color=DOT, lw=0.9, ls="--")
    a1.text(ff[3], 20 * np.log10(ideal) + 2, f"1/β = {ideal:.1f}", color=DOT,
            fontsize=7.5)
    a1.axvline(fp, color=MUTED, lw=0.8, ls=":")
    a1.axvline(f3, color=POS, lw=0.9, ls=":")
    a1.set_ylabel("gain  (dB)"); a1.legend(fontsize=7)
    a1.set_title(f"bandwidth ×(1+Aβ): {fp/1e3:.2f} kHz → {f3/1e6:.3f} MHz")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    lg = np.logspace(0, 5, 300)
    a2.loglog(lg, np.abs(1 / (1 + lg)) * 100, color=ORANGE, lw=1.7,
              label="gain error  ≈ 1/Aβ")
    a2.loglog(lg, np.abs(0.5 / (1 + lg)) * 100, color=PURP, lw=1.4,
              label="response to +50% in A")
    a2.axvline(loop, color=FG, lw=1.0, ls="--")
    a2.set_xlabel("loop gain  Aβ"); a2.set_ylabel("%")
    a2.legend(fontsize=7)
    a2.set_title("everything the loop encloses is divided by 1+Aβ")

    readout(fig, 0.845, 0.90, [
        "AMPLIFIER", "─" * 26,
        f"A0          {A0:>10.3e}",
        f"in dB       {A0_dB:>10.1f}dB",
        f"open-loop f {fp_kHz:>10.2f}kHz",
        "", "FEEDBACK", "─" * 26,
        f"β           {beta_f:>10.5f}",
        f"1/β ideal   {ideal:>10.4f}",
        f"loop gain   {loop:>10.1f}",
        f"in dB       {20*np.log10(loop):>10.2f}dB",
        "", "CLOSED LOOP", "─" * 26,
        f"Acl         {A0/(1+loop):>10.5f}",
        f"error       {err:>+10.4f}%",
        f"bandwidth   {f3/1e6:>10.4f}MHz",
        f"GBW check   {A0*fp/1e6:>10.3f}MHz",
        "", "DESENSITIVITY", "─" * 26,
        f"ΔA          {dA_pct:>+10.1f}%",
        f"ΔAcl        {dcl:>+10.4f}%",
        f"suppressed  {abs(dA_pct/max(abs(dcl),1e-9)):>10.1f}×",
        "", "distortion, Zout and",
        "gain error all divide",
        "by the same 1+Aβ",
    ], color=GREEN if loop > 100 else ORANGE)
    footer(fig, f"Acl = A/(1+Aβ) → 1/β   ·   loop gain {loop:,.0f}   ·   "
                f"a {dA_pct:+.0f}% device change becomes {dcl:+.4f}%")
    plt.show()


w5 = dict(A0_dB=widgets.FloatSlider(value=100, min=40, max=140, step=5,
                                    description="open-loop dB:", **SL),
          beta_pct=widgets.FloatSlider(value=1.0, min=0.05, max=100, step=0.05,
                                       description="β  (%):", **SL),
          fp_kHz=widgets.FloatSlider(value=0.1, min=0.01, max=10, step=0.01,
                                     description="pole (kHz):", **SL),
          dA_pct=widgets.FloatSlider(value=50, min=-90, max=200, step=10,
                                     description="ΔA  (%):", **SL))
display(widgets.HBox([w5["A0_dB"], w5["beta_pct"], w5["fp_kHz"], w5["dA_pct"]]),
        widgets.interactive_output(draw_feedback, w5))

## Cascading — and why the stages must be chosen to fit each other

Two stages in series multiply their gains, but only if the second does not load the first. What actually happens at the junction is the voltage divider from the theory notebook:

$$A_{total}=A_1\cdot\frac{R_{in,2}}{R_{out,1}+R_{in,2}}\cdot A_2$$

Two common-emitter stages illustrate the problem exactly. Each has $R_{out}\approx4.3$ kΩ and $R_{in}\approx3.9$ kΩ, so the interface throws away $3.9/(4.3+3.9)=0.48$ — nearly half the signal, more than 6 dB, before the second stage sees anything.

Insert a follower and the same two stages behave completely differently. The follower's 155 kΩ input barely loads the first stage, and its 26 Ω output barely troubles the second, so the interface loss falls from 6.4 dB to essentially zero. It contributes no gain of its own and yet more than doubles the total.

That is the argument for the whole chapter. A CE stage makes gain but has awkward impedances at both ends; a follower fixes impedances but makes no gain; a differential pair rejects common-mode but needs a current-source tail from the previous notebook. **None of them is useful alone**, and an op-amp is precisely the standard assembly of all three — which is where the next notebook starts.

In [ ]:
def draw_cascade(Ic_mA, Rc_k, use_buffer, RL_k):
    Ic, Rc, RL = Ic_mA * 1e-3, Rc_k * 1e3, RL_k * 1e3
    p = hybrid_pi(Ic)
    A1 = -p["gm"] * par(Rc, p["ro"])
    Rout1 = par(Rc, p["ro"])
    Rin2 = p["rpi"]
    A2 = -p["gm"] * par(Rc, p["ro"], RL)
    if use_buffer:
        Rin_b = p["rpi"] + (BETA + 1) * 1e3
        loss1 = Rin_b / (Rout1 + Rin_b)
        Rout_b = 1 / p["gm"] + Rout1 / (BETA + 1)
        loss2 = Rin2 / (Rout_b + Rin2)
        chain = [("stage 1", A1), ("interface", loss1), ("follower", 0.975),
                 ("interface", loss2), ("stage 2", A2)]
    else:
        loss1 = Rin2 / (Rout1 + Rin2)
        chain = [("stage 1", A1), ("interface", loss1), ("stage 2", A2)]
    total = 1.0
    for _, g in chain:
        total *= g
    ideal = A1 * A2

    fig = plt.figure(figsize=(13.0, 4.9))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.15, 0.55],
                          wspace=0.3, hspace=0.5, left=0.055, right=0.995,
                          top=0.88, bottom=0.13)

    a0 = panel(fig.add_subplot(gs[:, 0]), BLUE)
    names = [c[0] for c in chain]
    vals = [20 * np.log10(abs(c[1])) for c in chain]
    run = np.cumsum([0] + vals)
    for i, (nm, v) in enumerate(zip(names, vals)):
        a0.bar(i, v, bottom=run[i], width=0.62,
               color=NEG if v < 0 else (POS if "stage" in nm else PURP))
        a0.text(i, run[i] + v + (1.2 if v > 0 else -2.4), f"{v:+.2f}",
                ha="center", fontsize=7.5)
    a0.bar(len(chain), run[-1], width=0.62, color=DOT)
    a0.text(len(chain), run[-1] + 1.2, f"{run[-1]:+.2f}", ha="center", fontsize=8)
    a0.set_xticks(range(len(chain) + 1))
    a0.set_xticklabels(names + ["total"], rotation=25, ha="right", fontsize=7)
    a0.set_ylabel("dB")
    a0.set_title(f"where the gain goes — total {20*np.log10(abs(total)):.2f} dB")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    rr = np.logspace(2, 6, 300)
    a1.semilogx(rr / 1e3, rr / (Rout1 + rr), color=ORANGE, lw=1.8)
    a1.axvline(Rin2 / 1e3, color=NEG, lw=1.0, ls="--")
    a1.text(Rin2 / 1e3 * 1.1, 0.25, "CE input", color=NEG, fontsize=7)
    if use_buffer:
        a1.axvline((p["rpi"] + (BETA + 1) * 1e3) / 1e3, color=GREEN, lw=1.0,
                   ls="--")
        a1.text((p["rpi"] + (BETA + 1) * 1e3) / 1e3 * 0.35, 0.6, "follower input",
                color=GREEN, fontsize=7)
    a1.set_ylim(0, 1.05)
    a1.set_xlabel("next-stage input impedance  (kΩ)")
    a1.set_ylabel("fraction that survives")
    a1.set_title("the interface is just a voltage divider")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a2.bar([0, 1], [20 * np.log10(abs(ideal)), 20 * np.log10(abs(total))],
           color=[MUTED, DOT], width=0.5)
    for x, v in ((0, abs(ideal)), (1, abs(total))):
        a2.text(x, 20 * np.log10(v), f"  {20*np.log10(v):.2f} dB", ha="center",
                va="bottom", fontsize=8)
    a2.set_xticks([0, 1])
    a2.set_xticklabels(["A1·A2 if no loading", "what you actually get"],
                       fontsize=7.5)
    a2.set_ylabel("dB")
    a2.set_title(f"loading costs {20*np.log10(abs(ideal/total)):.2f} dB")

    readout(fig, 0.845, 0.88, [
        "STAGES", "─" * 26,
        f"Ic each     {Ic_mA:>10.3f}mA",
        f"Rc          {Rc_k:>10.2f}kΩ",
        f"load        {RL_k:>10.1f}kΩ",
        f"buffer      {str(bool(use_buffer)):>14s}",
        "", "IMPEDANCES", "─" * 26,
        f"Rout stage1 {Rout1/1e3:>10.3f}kΩ",
        f"Rin  stage2 {Rin2/1e3:>10.3f}kΩ",
        (f"Rin  buffer {(p['rpi']+(BETA+1)*1e3)/1e3:>10.1f}kΩ" if use_buffer
         else "no buffer"),
        (f"Rout buffer {1/p['gm']+Rout1/(BETA+1):>10.2f}Ω" if use_buffer
         else ""),
        "", "GAIN", "─" * 26,
        f"A1          {A1:>+10.2f}",
        f"A2          {A2:>+10.2f}",
        f"A1·A2       {ideal:>+10.1f}",
        f"actual      {total:>+10.1f}",
        f"in dB       {20*np.log10(abs(total)):>10.2f}dB",
        f"lost        {20*np.log10(abs(ideal/total)):>10.2f}dB",
        "", "a buffer adds no gain",
        "and doubles the total",
    ], color=GREEN if use_buffer else ORANGE)
    footer(fig, "A_total = A1 · [Rin2/(Rout1+Rin2)] · A2   ·   "
                "impedance mismatch is a divider, not a subtlety")
    plt.show()


w6 = dict(Ic_mA=widgets.FloatSlider(value=1.0, min=0.1, max=4.0, step=0.1,
                                    description="Ic (mA):", **SL),
          Rc_k=widgets.FloatSlider(value=4.7, min=0.5, max=15, step=0.1,
                                   description="Rc (kΩ):", **SL),
          use_buffer=widgets.Checkbox(value=False,
                                      description="insert a follower between them",
                                      indent=False),
          RL_k=widgets.FloatSlider(value=100, min=1, max=1000, step=1,
                                   description="load (kΩ):", **SL))
display(widgets.HBox([w6["Ic_mA"], w6["Rc_k"], w6["RL_k"], w6["use_buffer"]]),
        widgets.interactive_output(draw_cascade, w6))